In [1]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd


# -------------------------
# 1. Check probability validity
# -------------------------
def s_nan(y_prob, eps=1e-3):
    if not np.all(np.isfinite(y_prob)):
        return 0

    row_sums = np.sum(y_prob, axis=1)
    if np.all(np.abs(row_sums - 1.0) <= eps):
        return 1
    return 0


# -------------------------
# 2. Majority + JSD score
# -------------------------
def s_maj_jsd(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_true))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    # Majority score
    f_max = np.max(pred_dist)
    s_maj = 1 - (f_max - 1/n_classes) / (1 - 1/n_classes)

    # Jensen-Shannon divergence
    # jensenshannon() trả về căn bậc hai của JSD dùng log e
    js_distance = jensenshannon(true_dist, pred_dist)
    
    # Bình phương để lấy JSD (Divergence)
    js_divergence = js_distance**2
    s_jsd = 1 - (js_divergence / np.log(2))

    return s_maj, s_jsd


# -------------------------
# 3. Entropy score
# -------------------------
def s_ent(y_prob):
    N, K = y_prob.shape

    h = -np.sum(y_prob * np.log(y_prob + 1e-10), axis=1) / np.log(K)
    m = np.median(h)

    return 1 - min(abs(m - 0.5) / 0.5, 1.0)


# -------------------------
# 4. Drift score
# -------------------------
def s_drift(y_pred, ref_dist=None, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_pred))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)

    if ref_dist is None:
        ref_dist = np.ones(n_classes) / n_classes

    # jensenshannon trả về căn bậc hai của JSD (Distance)
    js_distance = jensenshannon(pred_dist, ref_dist)
    
    # Bình phương để có JSD (Divergence)
    js_divergence = js_distance**2
    
    return 1 - (js_divergence / np.log(2))


# -------------------------
# 5. Efficiency score
# -------------------------
def s_eff(mask):
    return np.mean(mask)


# -------------------------
# 6. Leakage score (generic probe)
# -------------------------
def s_leak_1(mask, y_true, probe_model, test_size=0.3, random_state=40):

    X_train, X_test, y_train, y_test = train_test_split(
        mask, y_true,
        test_size=test_size,
        random_state=random_state,
        stratify=y_true
    )

    probe_model.fit(X_train, y_train)

    if hasattr(probe_model, "predict_proba"):
        y_score = probe_model.predict_proba(X_test)
    else:
        y_score = probe_model.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score, multi_class="ovr")
    return 1 - auc


# -------------------------
# 7. Logistic leakage
# -------------------------
def s_leak(X, y, test_size=0.3, random_state=40):

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    probe = LogisticRegression(max_iter=1000)
    probe.fit(Xtr, ytr)

    y_score = probe.predict_proba(Xte)

    auc = roc_auc_score(yte, y_score, multi_class="ovr")
    return float(1 - auc)

# -------------------------
# 8. Final geometric mean score
# -------------------------
def S_san_plus(values):
    values = np.clip(values, 1e-10, 1)
    return np.prod(values) ** (1 / len(values))


In [2]:
def compute_ref_dist(y_train):
    y = y_train.astype(int).to_numpy()
    n_classes = len(np.unique(y))
    return np.bincount(y, minlength=n_classes) / len(y)

def run_sanity_evaluation(
    *,
    path_1,
    path_2,
    version_name,
    train_name,
    n_classes=3
):
    print(
        f"[START] Sanity evaluation | "
        f"train={train_name} | version={version_name}"
    )

    rows = []

    # -----------------
    # Load train
    # -----------------
    train_df = pd.read_csv(f"{path_1}/{train_name}.csv")
    y_train = train_df["label_3"]
    ref_dist = compute_ref_dist(y_train)

    # -----------------
    # Loop phases
    # -----------------
    for phase in [1, 2, 3, 4]:
        print(f"  -> Running phase {phase}...")

        # ---- probability matrix
        prob_df = pd.read_csv(
            f"{path_2}/probability_matrix_{version_name}_phase{phase}.csv"
        )

        y_true = prob_df["y_true"].astype(int).to_numpy()
        y_pred = prob_df["y_pred"].astype(int).to_numpy()
        y_prob = prob_df[
            ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
        ].to_numpy()

        # ---- test data (for leakage & efficiency)
        test_df = pd.read_csv(f"{path_1}/test_{phase}.csv")
        X_test = test_df.drop(columns=["label_3"], errors="ignore")
        mask = X_test.notna().astype(int).values

        # ---- metrics
        smaj, sjsd = s_maj_jsd(y_true, y_pred, n_classes=n_classes)

        snan   = s_nan(y_prob)
        sent   = s_ent(y_prob)
        sdrift = s_drift(y_pred, ref_dist, n_classes=n_classes)
        sleak  = s_leak(mask, y_true)
        s_eff_ = s_eff(mask)

        s_san_plus = S_san_plus([
            snan,
            sjsd,
            sent,
            sdrift,
            s_eff_,
            sleak
        ])

        rows.append({
            "train_name": train_name,
            "version_name": version_name,
            "phase": phase,
            "snan": snan,
            "smaj": smaj,
            "sjsd": sjsd,
            "sent": sent,
            "sdrift": sdrift,
            "sleak": sleak,
            "s_san+": s_san_plus,
            "s_eff": s_eff_
        })

    print(
        f"[END] Sanity evaluation DONE | "
        f"rows={len(rows)} | "
        f"phases=4"
    )

    columns_order = [
        "train_name",
        "version_name",
        "phase",
        "snan",
        "smaj",
        "sjsd",
        "sent",
        "sdrift",
        "sleak",
        "s_san+",
        "s_eff"
    ]

    return pd.DataFrame(rows)[columns_order]

# Chạy với các version data

## V1 (Median)

In [3]:
path_1 = "/kaggle/input/lo-dataset/Median/Median"

In [4]:
path_2 = "/kaggle/input/datasets/uyentran10/rnn-results/RNN_results"

In [5]:
df_v1 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V1 (Median)",
    train_name="train_median"
)
df_v1

[START] Sanity evaluation | train=train_median | version=V1 (Median)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median,V1 (Median),1,1,0.001168,0.999144,0.005447,0.999145,0.5,0.373595,1.0
1,train_median,V1 (Median),2,1,0.001174,0.999147,0.005052,0.999147,0.5,0.368929,1.0
2,train_median,V1 (Median),3,1,0.002594,0.999626,0.009562,0.999626,0.5,0.410394,1.0
3,train_median,V1 (Median),4,1,0.004420,0.999968,0.000093,0.999968,0.5,0.189634,1.0


In [6]:
df_v1.to_csv("results_v1.csv", index=False)

## V2 (Median CDSMOTE)

In [7]:
df_v2 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V2 (Median CDS)",
    train_name="train_median_cdsmote"
)
df_v2

[START] Sanity evaluation | train=train_median_cdsmote | version=V2 (Median CDS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_cdsmote,V2 (Median CDS),1,1,0.016423,0.998084,0.002443,0.574921,0.5,0.298042,1.0
1,train_median_cdsmote,V2 (Median CDS),2,1,0.014667,0.998557,0.001985,0.572465,0.5,0.287717,1.0
2,train_median_cdsmote,V2 (Median CDS),3,1,0.040892,0.992029,0.007264,0.609542,0.5,0.360538,1.0
3,train_median_cdsmote,V2 (Median CDS),4,1,0.012893,0.999220,0.000060,0.571661,0.5,0.160606,1.0


In [8]:
df_v2.to_csv("results_v2.csv", index=False)

## V3 (Median SASMOTE)

In [9]:
df_v3 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V3 (Median SAS)",
    train_name="train_median_sasmote"
)
df_v3

[START] Sanity evaluation | train=train_median_sasmote | version=V3 (Median SAS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_sasmote,V3 (Median SAS),1,1,0.021514,0.997404,0.001185,0.585573,0.5,0.264960,1.0
1,train_median_sasmote,V3 (Median SAS),2,1,0.019101,0.997991,0.000894,0.582233,0.5,0.252586,1.0
2,train_median_sasmote,V3 (Median SAS),3,1,0.024469,0.996711,0.001295,0.590284,0.5,0.269253,1.0
3,train_median_sasmote,V3 (Median SAS),4,1,0.011628,0.999420,0.000008,0.569309,0.5,0.115366,1.0


In [10]:
df_v3.to_csv("results_v3.csv", index=False)

## V4 (Median RadiusSMOTE)

In [11]:
df_v4 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V4 (Median Radius)",
    train_name="train_median_radiussmote"
)
df_v4

[START] Sanity evaluation | train=train_median_radiussmote | version=V4 (Median Radius)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_radiussmote,V4 (Median Radius),1,1,0.014338,0.998646,0.021319,0.572011,0.5,0.427324,1.0
1,train_median_radiussmote,V4 (Median Radius),2,1,0.018901,0.997628,0.042936,0.579127,0.5,0.481122,1.0
2,train_median_radiussmote,V4 (Median Radius),3,1,0.019055,0.997820,0.039257,0.580510,0.5,0.474196,1.0
3,train_median_radiussmote,V4 (Median Radius),4,1,0.007853,0.999870,0.000077,0.561639,0.5,0.166828,1.0


In [12]:
df_v4.to_csv("results_v4.csv", index=False)

## V5 (Mean)

In [13]:
path_1 = "/kaggle/input/lo-dataset/Mean/Mean"

In [14]:
df_v5 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V5 (Mean)",
    train_name="train_mean"
)
df_v5

[START] Sanity evaluation | train=train_mean | version=V5 (Mean)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean,V5 (Mean),1,1,0.000561,0.998842,0.000574,0.998843,0.5,0.256762,1.0
1,train_mean,V5 (Mean),2,1,0.000594,0.998863,0.000636,0.998864,0.5,0.261176,1.0
2,train_mean,V5 (Mean),3,1,0.001110,0.999198,0.000940,0.999198,0.5,0.278764,1.0
3,train_mean,V5 (Mean),4,1,0.004504,0.999972,0.000011,0.999972,0.5,0.133095,1.0


In [15]:
df_v5.to_csv("results_v5.csv", index=False)

## V6 (Mean CDSMOTE)

In [16]:
df_v6 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V6 (Mean CDS)",
    train_name="train_mean_cdsmote"
)
df_v6

[START] Sanity evaluation | train=train_mean_cdsmote | version=V6 (Mean CDS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_cdsmote,V6 (Mean CDS),1,1,0.012080,0.999311,0.001213,0.570481,0.5,0.264938,1.0
1,train_mean_cdsmote,V6 (Mean CDS),2,1,0.012015,0.999299,0.001037,0.570422,0.5,0.258075,1.0
2,train_mean_cdsmote,V6 (Mean CDS),3,1,0.014932,0.998838,0.002069,0.575806,0.5,0.290016,1.0
3,train_mean_cdsmote,V6 (Mean CDS),4,1,0.012093,0.999349,0.000060,0.569915,0.5,0.160488,1.0


In [17]:
df_v6.to_csv("results_v6.csv", index=False)

## V7 (Mean SASMOTE)

In [18]:
df_v7 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V7 (Mean SAS)",
    train_name="train_mean_sasmote"
)
df_v7

[START] Sanity evaluation | train=train_mean_sasmote | version=V7 (Mean SAS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_sasmote,V7 (Mean SAS),1,1,0.030845,0.994981,0.006822,0.598334,0.5,0.355857,1.0
1,train_mean_sasmote,V7 (Mean SAS),2,1,0.030025,0.995197,0.006438,0.597209,0.5,0.352336,1.0
2,train_mean_sasmote,V7 (Mean SAS),3,1,0.027580,0.995966,0.004651,0.595375,0.5,0.333619,1.0
3,train_mean_sasmote,V7 (Mean SAS),4,1,0.012448,0.999294,0.000043,0.570772,0.5,0.151833,1.0


In [19]:
df_v7.to_csv("results_v7.csv", index=False)

## V8 (Mean RadiusSMOTE)

In [20]:
df_v8 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V8 (Mean Radius)",
    train_name="train_mean_radiussmote"
)
df_v8

[START] Sanity evaluation | train=train_mean_radiussmote | version=V8 (Mean Radius)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_radiussmote,V8 (Mean Radius),1,1,0.129304,0.964525,0.282544,0.700573,0.5,0.676037,1.0
1,train_mean_radiussmote,V8 (Mean Radius),2,1,0.072157,0.983120,0.159189,0.656187,0.5,0.609658,1.0
2,train_mean_radiussmote,V8 (Mean Radius),3,1,0.091612,0.976923,0.190546,0.669046,0.5,0.629575,1.0
3,train_mean_radiussmote,V8 (Mean Radius),4,1,0.010873,0.999528,0.000132,0.567444,0.5,0.182795,1.0


In [21]:
df_v8.to_csv("results_v8.csv", index=False)

## V9 (Extra Trees)

In [22]:
path_1 = "/kaggle/input/lo-dataset/Extra_trees/Extra_trees"

In [23]:
df_v9 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V9 (Extra)",
    train_name="train_extra"
)
df_v9

[START] Sanity evaluation | train=train_extra | version=V9 (Extra)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra,V9 (Extra),1,1,0.015894,0.997639,0.017659,0.997639,0.5,0.454269,1.0
1,train_extra,V9 (Extra),2,1,0.005898,0.999422,0.007535,0.999422,0.5,0.394392,1.0
2,train_extra,V9 (Extra),3,1,0.003536,0.999909,0.003311,0.999909,0.5,0.343931,1.0
3,train_extra,V9 (Extra),4,1,0.004394,0.999977,0.000027,0.999977,0.5,0.153857,1.0


In [24]:
df_v9.to_csv("results_v9.csv", index=False)

## V10 (Extra Trees CDSMOTE)

In [25]:
df_v10 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V10 (Extra CDS)",
    train_name="train_extra_cdsmote"
)
df_v10

[START] Sanity evaluation | train=train_extra_cdsmote | version=V10 (Extra CDS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_cdsmote,V10 (Extra CDS),1,1,0.010492,0.998986,1.430448e-04,0.564378,0.5,0.185180,1.0
1,train_extra_cdsmote,V10 (Extra CDS),2,1,0.008899,0.999279,2.896934e-05,0.561598,0.5,0.141798,1.0
2,train_extra_cdsmote,V10 (Extra CDS),3,1,0.016358,0.998575,1.181188e-04,0.577720,0.5,0.180052,1.0
3,train_extra_cdsmote,V10 (Extra CDS),4,1,0.010441,0.999592,9.246870e-07,0.566928,0.5,0.079993,1.0


In [26]:
df_v10.to_csv("results_v10.csv", index=False)

## V11 (Extra Trees SASMOTE)

In [27]:
df_v11 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V11 (Extra SAS)",
    train_name="train_extra_sasmote"
)
df_v11

[START] Sanity evaluation | train=train_extra_sasmote | version=V11 (Extra SAS)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_sasmote,V11 (Extra SAS),1,1,0.002684,0.999783,2.060299e-05,0.549300,0.5,0.133486,1.0
1,train_extra_sasmote,V11 (Extra SAS),2,1,0.002581,0.999721,1.926899e-05,0.548778,0.5,0.131983,1.0
2,train_extra_sasmote,V11 (Extra SAS),3,1,0.007059,0.999871,1.620901e-05,0.560051,0.5,0.128672,1.0
3,train_extra_sasmote,V11 (Extra SAS),4,1,0.009454,0.999717,2.797654e-07,0.564925,0.5,0.065505,1.0


In [28]:
df_v11.to_csv("results_v11.csv", index=False)

## V12 (Extra Trees RadiusSMOTE)

In [29]:
df_v12 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V12 (Extra Radius)",
    train_name="train_extra_radiussmote"
)
df_v12

[START] Sanity evaluation | train=train_extra_radiussmote | version=V12 (Extra Radius)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_radiussmote,V12 (Extra Radius),1,1,0.007143,0.999281,0.002370,0.559901,0.5,0.295286,1.0
1,train_extra_radiussmote,V12 (Extra Radius),2,1,0.005485,0.999753,0.001867,0.556442,0.5,0.283522,1.0
2,train_extra_radiussmote,V12 (Extra Radius),3,1,0.005982,0.999990,0.000638,0.557366,0.5,0.237132,1.0
3,train_extra_radiussmote,V12 (Extra Radius),4,1,0.008169,0.999847,0.000019,0.562289,0.5,0.131760,1.0


In [30]:
df_v12.to_csv("results_v12.csv", index=False)